# 1. Data Ingestion

In [27]:
import pyodbc
import numpy as np
import pandas as pd

In [28]:
customers = pd.read_csv('olist_data/olist_customers_dataset.csv')
geolocation = pd.read_csv('olist_data/olist_geolocation_dataset.csv')
order_items = pd.read_csv('olist_data/olist_order_items_dataset.csv')
order_payments = pd.read_csv('olist_data/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('olist_data/olist_order_reviews_dataset.csv')
orders_dataset = pd.read_csv('olist_data/olist_orders_dataset.csv')
products = pd.read_csv('olist_data/olist_products_dataset.csv')
sellers = pd.read_csv('olist_data/olist_sellers_dataset.csv')
product_category_name_translation = pd.read_csv('olist_data/product_category_name_translation.csv')

# 2. Check Duplicates& Missing Values

In [29]:
datasets = {
    'customers':customers,
    'geolocation':geolocation,
    'order_items':order_items,
    'order_payments':order_payments,
    'order_reviews':order_reviews,
    'orders_dataset':orders_dataset,
    'products':products,
    'sellers':sellers,
    'product_category_name_translation':product_category_name_translation
}

In [30]:
for name, set in datasets.items():
    duplicate_count = set.duplicated().sum()
    if duplicate_count:
        print (f'--{name.upper()} have {duplicate_count} duplicates.')
        set.drop_duplicates(inplace=True)
        print (f'--Droped {duplicate_count} rows from {name.upper()}.')
    else:
        print (f'-{name.upper()} have no duplicates.')
    print('=' * 49)

-CUSTOMERS have no duplicates.
--GEOLOCATION have 261831 duplicates.
--Droped 261831 rows from GEOLOCATION.
-ORDER_ITEMS have no duplicates.
-ORDER_PAYMENTS have no duplicates.
-ORDER_REVIEWS have no duplicates.
-ORDERS_DATASET have no duplicates.
-PRODUCTS have no duplicates.
-SELLERS have no duplicates.
-PRODUCT_CATEGORY_NAME_TRANSLATION have no duplicates.


In [31]:
for name, set in datasets.items():
    null_rows = set.isna().sum()
    missing_vals = null_rows[null_rows > 0]
    print (f'========== {name.upper()} Missing Values ==========')
    if not missing_vals.empty:
        print(missing_vals)
        print(f'\n-Missing Values Percents: \n{round(missing_vals / set.shape[0] * 100, 2)}')
    else:
        print ('NO Missing Values.')
    print('_' * 49+'\n')


========== CUSTOMERS Missing Values ==========
NO Missing Values.
_________________________________________________

========== GEOLOCATION Missing Values ==========
NO Missing Values.
_________________________________________________

========== ORDER_ITEMS Missing Values ==========
NO Missing Values.
_________________________________________________

========== ORDER_PAYMENTS Missing Values ==========
NO Missing Values.
_________________________________________________

========== ORDER_REVIEWS Missing Values ==========
review_comment_title      87656
review_comment_message    58247
dtype: int64

-Missing Values Percents: 
review_comment_title      88.34
review_comment_message    58.70
dtype: float64
_________________________________________________

========== ORDERS_DATASET Missing Values ==========
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

-Missing Values Percents: 
order_approved_at             

In [32]:
orders_dataset[orders_dataset.order_delivered_customer_date.isna()].groupby('order_status').order_id.count()
# Missing dates align with natural order status so it won't be removed because deleting them would affect the analysis ahead 

order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_id, dtype: int64

# 3. Handling Missing values

## 3.1 Products

In [33]:
# Products are linked to real sales and revenue so we keep the null names with changeing their values to 'unknown'
products.fillna({'product_category_name':'unknown'}, inplace = True)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [34]:
# Fill missing numbers (weights, sizes, photo quantities) with 0 to keep data types consistent.
# We can't fill data with mean/ median becasue every product is different and there is too many products for Group-wise (Subgroup) Imputation
empty_product_numeric_cols = ['product_name_lenght',
                                'product_description_lenght',
                                'product_photos_qty',
                                'product_weight_g',
                                'product_length_cm',
                                'product_height_cm',
                                'product_width_cm']
for col in empty_product_numeric_cols:
    products.fillna({col:0}, inplace=True)
products.isna().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

## 3.2 Order_Reviews

In [35]:
# We won't drop the nulls in the reviews because we need the product scores we can just fill them
order_reviews.fillna({'review_comment_title':'No Title',
                    'review_comment_message':'No Message'}, inplace=True)
order_reviews.isna().sum()

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

# 4. Data Types

In [36]:
for name, set in datasets.items():
    print (f'========== {name.upper()} Data Types ==========')
    print(set.dtypes)
    print('_' * 49+'\n')

========== CUSTOMERS Data Types ==========
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
_________________________________________________

========== GEOLOCATION Data Types ==========
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object
_________________________________________________

========== ORDER_ITEMS Data Types ==========
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object
_________________________________________________

========== ORDER_PAYMENTS Data Types ==========
order_id                    str
payment_se

## 4.1 Fix The Dates Data types To Enable Time-based Analysis

In [37]:
order_items.shipping_limit_date = pd.to_datetime(order_items.shipping_limit_date)
order_items.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

In [38]:
order_reviews.review_answer_timestamp = pd.to_datetime(order_reviews.review_answer_timestamp)
order_reviews.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date                  str
review_answer_timestamp    datetime64[us]
dtype: object

In [39]:
str_to_date = ['order_purchase_timestamp',
                'order_approved_at',
                'order_delivered_carrier_date',
                'order_delivered_customer_date',
                'order_estimated_delivery_date']
for date in str_to_date:
    orders_dataset[date] = pd.to_datetime(orders_dataset[date])
orders_dataset.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

# 5. Assessing the data size

In [40]:
for name, set in datasets.items():
    print (f'========== {name.upper()} Shape ==========')
    print(set.shape)
    print('_' * 40+'\n')

========== CUSTOMERS Shape ==========
(99441, 5)
________________________________________

========== GEOLOCATION Shape ==========
(738332, 5)
________________________________________

========== ORDER_ITEMS Shape ==========
(112650, 7)
________________________________________

========== ORDER_PAYMENTS Shape ==========
(103886, 5)
________________________________________

========== ORDER_REVIEWS Shape ==========
(99224, 7)
________________________________________

========== ORDERS_DATASET Shape ==========
(99441, 8)
________________________________________

========== PRODUCTS Shape ==========
(32951, 9)
________________________________________

========== SELLERS Shape ==========
(3095, 4)
________________________________________

========== PRODUCT_CATEGORY_NAME_TRANSLATION Shape ==========
(71, 2)
________________________________________



## 5.1 Compressing Geolocation data 
Because it is almost 1M rows
it has the coordinates of alot of similar locations
That would be too much to export to SQL

In [41]:
geo_cleaned = geolocation.groupby('geolocation_zip_code_prefix').agg({'geolocation_lat':'mean',
                                                        'geolocation_lng':'mean',
                                                        'geolocation_city':'first',
                                                        'geolocation_state':'first'}).reset_index()

print(f'Compressed geolocation: {len(geolocation)} rows To geo_cleaned: {len(geo_cleaned)}')

geo_cleaned.columns = ['zip_code_prefix', 'lat', 'lng', 'city', 'state']

geo_cleaned.head()

Compressed geolocation: 738332 rows To geo_cleaned: 19015


,zip_code_prefix,lat,lng,city,state
0,1001,-23.550227,-46.634039,sao paulo,SP
1,1002,-23.547657,-46.634991,sao paulo,SP
2,1003,-23.549000,-46.635582,sao paulo,SP
3,1004,-23.549829,-46.634792,sao paulo,SP
4,1005,-23.549547,-46.636406,sao paulo,SP


# 6. Exporting to SQL

In [42]:
SQL_SERVER = r"localhost"
SQL_DATABASE = "Olist"

connection_string = (
    r"DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DATABASE};"
    r"Trusted_Connection=yes;"
    r"Encrypt=yes;"
    r"TrustServerCertificate=yes;"
)

In [43]:
tables_to_export = {
    "customer": customers,
    "geolocation": geo_cleaned,
# Export the aggregated geolocation DataFrame created above.
    "order_item": order_items,
    "order_payment": order_payments,
    "order_review": order_reviews,
    "order": orders_dataset,
    "product": products,
    "seller": sellers,
    "product_category_translation": product_category_name_translation,
}

In [44]:
def sql_type(series):
    dtype = series.dtype

    if pd.api.types.is_datetime64_any_dtype(dtype):
        return "DATETIME2"
    if pd.api.types.is_bool_dtype(dtype):
        return "BIT"
    if pd.api.types.is_integer_dtype(dtype):
        return "BIGINT"
    if pd.api.types.is_float_dtype(dtype):
        return "FLOAT"

    return "NVARCHAR(MAX)"

In [45]:
def sql_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    if isinstance(value, np.generic):
        return value.item()
    return value

In [46]:
connection = pyodbc.connect(connection_string)
cursor = connection.cursor()
cursor.fast_executemany = True

try:
    for table_name, frame in tables_to_export.items():
        columns = list(frame.columns)

        column_definitions = ", ".join(
            f"[{column}] {sql_type(frame[column])} NULL"
            for column in columns
        )

        cursor.execute(
            f"IF OBJECT_ID(N'dbo.{table_name}', N'U') IS NOT NULL "
            f"DROP TABLE dbo.[{table_name}]"
        )
        cursor.execute(
            f"CREATE TABLE dbo.[{table_name}] ({column_definitions})"
        )

        quoted_columns = ", ".join(f"[{column}]" for column in columns)
        placeholders = ", ".join("?" for _ in columns)
        insert_sql = (
            f"INSERT INTO dbo.[{table_name}] "
            f"({quoted_columns}) VALUES ({placeholders})"
        )

        batch_size = 5000
        for start in range(0, len(frame), batch_size):
            batch = frame.iloc[start : start + batch_size]
            rows = [
                tuple(sql_value(value) for value in row)
                for row in batch.itertuples(index=False, name=None)
            ]
            cursor.executemany(insert_sql, rows)

        print(f"Loaded dbo.{table_name}: {len(frame):,} rows")

    connection.commit()
    print("All nine tables exported successfully.")

except Exception:
    connection.rollback()
    raise

finally:
    cursor.close()
    connection.close()

Loaded dbo.customer: 99,441 rows
Loaded dbo.geolocation: 19,015 rows
Loaded dbo.order_item: 112,650 rows
Loaded dbo.order_payment: 103,886 rows
Loaded dbo.order_review: 99,224 rows
Loaded dbo.order: 99,441 rows
Loaded dbo.product: 32,951 rows
Loaded dbo.seller: 3,095 rows
Loaded dbo.product_category_translation: 71 rows
All nine tables exported successfully.
